In [1]:
from aligner import GeneLoader
from genetic_algorithm import NSGA_II
from HMM import HMM
from sentence_transformers import SentenceTransformer
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
import numpy as np

In [2]:
LOADER = GeneLoader("hg38_primary")

In [3]:
HMM_MODEL = HMM(n_states=len(LOADER.states), k=6, n_bases=6, model_name="supervised_10k_oldschema", fresh_start=True)

In [4]:
dna_bert_2_tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)
dna_bert_2_model = AutoModel.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

/home/ruminator/.cache/huggingface/modules/transformers_modules/zhihan1996/DNABERT_hyphen_2_hyphen_117M/7bce263b15377fc15361f52cfab88f8b586abda0/bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
objectives = {
    "cos_dist": {"reverse": False},
    "ham_dist": {"reverse": False},
    "hmm_prob": {"reverse": False}
}
GA = NSGA_II("test", objectives, 
             hmm=HMM_MODEL, 
             loader=LOADER, 
             pop_cap=2, 
             idx=21,
             model=dna_bert_2_model, 
             tokenizer=dna_bert_2_tokenizer
             )

In [ ]:
GA.popn.clear()
GA.load_popn()
GA.get_embeddings()
for i in range(50):
    GA.random_mutate_entry(0)
    GA.random_mutate_entry(1)
GA.crossover(0, 1, 2)
GA.get_mod_ll()
GA.get_mod_conv_score()
GA.get_embeddings()
GA.compute_total_fitness()
GA.set_update_flag()
with np.printoptions(threshold=300, edgeitems=3):
    for entry in GA.popn:
        print(entry["fitness"])
GA.popn.clear()

states [5 5 5 5 5 5 5 5 5 5 5 5 5 3 7 5 8 6 6 6 6 4 6 6 4 6 6 6 6 6 6 6 3 6 6 6 8
 6 6 6 6 6 2 1 6 6 6 6 6 6 8 6 6 8 6 6 6 6 6 6 6 7 6 6 6 3 6 6 7 6 6 6 6 6
 3 6 6 6 6 6 2 6 6 6 6 6 6 2 6 6 6 3 2 6 6 6 6 6 6 6 6 6 7 6 6 6 2 7 6 6 6
 6 6 6 8 6 6 6 7 6 8 8 6 6 8 6 6 6 1 8 1 3 8 6 6 6 6 6 6 6 6 6 6 6 6 1 6 1
 6 6 6 6 6 6 6 6 6 6 8 8 2 1 6 6 7 6 1 8 3 6 4 6 6 8 6 6 6 6 6 6 2 6 6 6 8
 6 6 8 6 1 5 5 5 5 5 5 7 5 5 5 5 5 5 5 5 5 6]
muts   [0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1
 0 0 0 0 0 1 1 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 0 1 0 0 0 0 0
 1 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 1 0 0 0 1 1 0 0 0
 0 0 0 1 0 0 0 1 0 1 1 0 0 1 0 0 0 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1
 0 0 0 0 0 0 0 0 0 0 1 1 1 1 0 0 1 0 1 1 1 0 1 0 0 1 0 0 0 0 0 0 1 0 0 0 1
 0 0 1 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1]
states [5 5 5 5 5 5 5 5 5 3 5 5 5 5 5 8 5 6 1 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 3 3
 8 4 6 6 6 6 6 6 6 6 6 6 6 6 8 6 6 6 6 6 1 6 1 6 6 6 6 3 6 6 8